# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jaymr06/flyrank/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Freestyle — AI Referral Opportunity**

Most content prioritization work focuses on traditional search signals: impressions, position, CTR. But AI assistants have introduced a second discovery channel where users click through to a page after an AI tool surfaces it — measured here as `ai_sessions_90d` and `ai_traffic_pct`. This channel is early and the data is sparse, but that sparsity is itself meaningful: it means the pages that *do* attract AI-referred traffic stand out, and understanding what they have in common could help content editors know which pages are worth improving for AI discoverability.

I am choosing this lane because it is an observational EDA and ranking problem, not a binary classification task. The guide is explicit that a classifier on sparse AI-session data would be statistically worthless. The honest version of this question — what broad content patterns appear around AI-referred traffic, and where might AI visibility be improved? — is a well-scoped 7-week project: explore the signal, identify patterns, produce a ranked opportunity list, and say carefully what the data can and cannot prove.

In [ ]:
import pandas as pd

# Load starter dataset — works from cloned repo or Colab (see SETUP.md)
try:
    df = pd.read_csv('../../data/raw/content_refresh_anonymized.csv')
except FileNotFoundError:
    import subprocess, sys
    subprocess.run(['git', 'clone', '--quiet',
                    'https://github.com/jaymr06/flyrank.git', 'flyrank'], check=True)
    df = pd.read_csv('flyrank/data/raw/content_refresh_anonymized.csv')

print(f"Dataset: {len(df):,} rows x {df.shape[1]} columns, {df['client_id'].nunique()} clients")
print(f"AI-related columns present: {[c for c in df.columns if 'ai' in c.lower()]}")


## 2. The question: decision, action, cost of a wrong call

**The question:** What broad content patterns are associated with AI-referred click-through traffic, and which pages are the best candidates to review for AI-readiness improvements?

**Unit of analysis:** One content item (page) — one row in the starter dataset.

**The decision:** A content editor choosing which pages to spend time on for AI-discoverability improvements, given limited review capacity.

**The action:** Structural and metadata edits — adding FAQ blocks, clear headings, concise definitions, explicit source attributions — that make a page more likely to be clicked through when an AI assistant surfaces it.

**The output:** A ranked list of candidate pages, ordered by observed AI-traffic signals and content characteristics, with reason codes a reviewer can inspect.

**Cost of a wrong recommendation:**
- *False positive (flag a page that doesn't need review):* wasted editor hours on a page that is already well-structured or has no realistic AI traffic opportunity.
- *False negative (miss a page worth reviewing):* a page that AI tools are already trying to surface, but with poor click-through, goes untouched — a missed channel.

Because editor time is the scarce resource, precision at the top of the ranked list matters more than overall recall. A reviewer working through the top 20–50 candidates needs those to be real opportunities, not noise.

**Why data helps here:** There are too many pages across too many clients for a human to prioritize by instinct. The observed signals (impressions, position, engagement, ai_sessions_90d, ai_traffic_pct, content type) span dozens of dimensions. A data-driven ranking can surface the right candidates faster than a manual scan — as long as it is honest about what it measured.

In [ ]:
# Review-scope sanity check: how many pages would land in the candidate pool?
n_total = len(df)
n_ai    = (df['ai_sessions_90d'] > 0).sum()
n_zero  = n_total - n_ai

print(f"Total pages in dataset:          {n_total:,}")
print(f"Pages with any AI sessions (90d): {n_ai:,}  ({n_ai/n_total*100:.1f}%)")
print(f"Pages with zero AI sessions:      {n_zero:,}  ({n_zero/n_total*100:.1f}%)")
print()
print("The ranking task: order all pages so editors reviewing the top-K see the most promising candidates first.")


## 3. Quick look at the data (2-3 real numbers)

Three numbers from the starter dataset that make this lane worth 7 weeks:

1. **Sparsity confirms the right method.** Only a small fraction of pages have any AI-referred sessions in 90 days. That rules out binary classification (as the guide warns) and points squarely to EDA and ranked scoring — the right approach for this lane.

2. **When AI traffic shows up, it is a non-trivial share.** Among pages that do have AI sessions, the median `ai_traffic_pct` is not negligible. That means the signal, when present, is real — not just a stray click.

3. **Visibility context differs.** Pages with AI sessions tend to sit at a different median search position than pages without. That directional difference is the first hint that search visibility and AI click-through are connected — and gives us a starting feature for the ranking model.

See the code cell below for the exact numbers.

In [ ]:
import numpy as np

ai    = df[df['ai_sessions_90d'] > 0].copy()
no_ai = df[df['ai_sessions_90d'] == 0].copy()

# --- Number 1: sparsity ---
print("=== 1. AI session sparsity ===")
print(f"{len(ai):,} of {len(df):,} pages ({len(ai)/len(df)*100:.1f}%) have >= 1 AI-referred session in 90d")
print(f"This rules out binary classification — too few positives for a reliable classifier.")
print()

# --- Number 2: ai_traffic_pct distribution among pages that have it ---
print("=== 2. AI traffic share among pages with AI sessions ===")
print(f"Median ai_traffic_pct : {ai['ai_traffic_pct'].median():.2f}%")
print(f"Mean   ai_traffic_pct : {ai['ai_traffic_pct'].mean():.2f}%")
print(f"Max    ai_sessions_90d: {ai['ai_sessions_90d'].max():.0f} sessions")
print("(ai_traffic_pct can exceed 100 — see data dictionary; this is expected, not an error)")
print()

# --- Number 3: search visibility comparison ---
print("=== 3. Search position: pages WITH vs WITHOUT AI sessions ===")
ai_pos    = ai[ai['avg_position'] > 0]['avg_position']      # 0 means no data
no_ai_pos = no_ai[no_ai['avg_position'] > 0]['avg_position']
print(f"Median avg_position — WITH AI sessions:    {ai_pos.median():.1f}")
print(f"Median avg_position — WITHOUT AI sessions: {no_ai_pos.median():.1f}")
print("(lower number = better ranking; directional only — not a causal claim)")


## 4. Careful words: what I can and can't claim

**What this work can say:**

- "We *observed* that X% of pages in the starter dataset received at least one AI-referred click-through in the 90-day window."
- "Pages with AI sessions *tended to* sit at a different median search position — a directional association in the observed data, not a causal finding."
- "This analysis produces a *decision-support* ranking: a list of pages ordered by their observed AI-traffic signals and content characteristics, for a human editor to review and act on."
- "The patterns are *measured* from click-through events — a user clicked from an AI assistant to a page. This says nothing about pages that were cited but not clicked, or about how any AI platform evaluates content internally."

**What this work cannot say:**

- That making content changes *will* increase AI traffic. No experiment was run; this data shows correlation, not causation.
- That pages without AI sessions are *not* being cited or surfaced by AI tools. `ai_sessions_90d` measures click-throughs, not mentions, citations, or AI rankings.
- Anything about how Google, Perplexity, ChatGPT, or any specific AI platform decides what to surface. The data does not expose that.
- That findings from the 30k-row starter slice will generalize to the full warehouse or to clients not in this dataset without re-validation.
- That a high `ai_traffic_pct` is inherently good — the denominator (`sessions_90d`) can be very small, making the ratio noisy at low volumes.

In [ ]:
# Sanity-check the AI columns before drawing any conclusions from them
print("ai_sessions_90d — value summary:")
print(df['ai_sessions_90d'].describe().round(2))
print()
print("ai_traffic_pct — value summary:")
print(df['ai_traffic_pct'].describe().round(2))
print()

# Flag any rows where ai_traffic_pct > 100 so we know it's real, not a bug
over_100 = (df['ai_traffic_pct'] > 100).sum()
print(f"Rows where ai_traffic_pct > 100: {over_100:,}  (expected per data dictionary — not an error)")


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.